# Alumni_Donor_Propensity_Forecaster_train

## CELL 1 — Import Libraries

This cell imports all the necessary Python libraries for data manipulation, machine learning model building, and evaluation.

In [2]:
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For creating static, interactive, and animated visualizations
import seaborn as sns  # For statistical data visualization

# Scikit-learn modules for preprocessing, model selection, and models
from sklearn.model_selection import train_test_split  # To split data into training and testing sets
from sklearn.compose import ColumnTransformer  # To apply different transformations to different columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # For scaling numerical and encoding categorical features
from sklearn.pipeline import Pipeline  # To create a pipeline of transformations and a model
from sklearn.linear_model import LogisticRegression  # Logistic Regression model
from sklearn.ensemble import RandomForestClassifier  # Random Forest Classifier model
from sklearn.metrics import (  # Metrics for model evaluation
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import joblib  # For saving and loading Python objects, especially scikit-learn models

# Specific ML libraries
from xgboost import XGBClassifier  # XGBoost Classifier model
from catboost import CatBoostClassifier  # CatBoost Classifier model

print("✅ Libraries imported successfully!")

ModuleNotFoundError: No module named 'catboost'

## CELL 2 — Upload and Load Dataset

This cell handles the upload of the `alumni_dataset.csv` file from your local machine to Google Colab, then loads it into a pandas DataFrame. It also displays basic information about the dataset.

In [ ]:
from google.colab import files  # Import the files module for uploading files in Colab

print("Please upload alumni_dataset.csv")  # Instruct the user to upload the specific file

uploaded = files.upload()  # Initiate the file upload dialog

# Get the name of the first uploaded file
file_name = next(iter(uploaded))

# Read the uploaded CSV file into a pandas DataFrame
data = pd.read_csv(file_name)

print("\n✅ Dataset loaded successfully!")  # Confirm successful loading
print("File name:", file_name)  # Display the name of the loaded file
print("Dataset shape:", data.shape)  # Display the dimensions of the DataFrame

print("\nColumn names:") # Display column names
print(data.columns.tolist())

display(data.head())  # Display the first 5 rows of the DataFrame to preview the data

Please upload alumni_dataset.csv


## CELL 3 — Data Validation

This cell performs basic data validation checks, including looking for missing values, duplicate rows, inspecting data types, and checking the distribution of the target variable.

In [ ]:
print("Missing Values:")  # Header for missing values check
print(data.isnull().sum().sum())  # Calculate and print the total count of missing values in the DataFrame

print("\nDuplicate Rows:")  # Header for duplicate rows check
print(data.duplicated().sum())  # Calculate and print the total count of duplicate rows in the DataFrame

print("\nData Types:")  # Header for data types check
print(data.info()) # Display data types for each column

print("\nTarget Distribution ('donated_next_12_months'):")  # Header for target variable distribution
print(data["donated_next_12_months"].value_counts())  # Show the distribution of values in the 'donated_next_12_months' column

## CELL 4 — Feature Engineering

This cell creates new features from the existing dataset to potentially improve model performance. These features are designed to capture more insights related to alumni engagement and donation propensity.

In [ ]:
# Calculate 'years_since_graduation' based on 'graduation_year'
data["years_since_graduation"] = (
    2026 - data["graduation_year"]
)

# Calculate 'email_open_rate', handling division by zero
data["email_open_rate"] = np.where(
    data["emails_received"] > 0,
    data["emails_opened"] / data["emails_received"],
    0
)

# Clip 'email_open_rate' to be between 0 and 1
data["email_open_rate"] = (
    data["email_open_rate"].clip(0, 1)
)

# Calculate 'average_donation_amount', handling division by zero
data["average_donation_amount"] = np.where(
    data["previous_donations"] > 0,
    data["total_donation_amount"] /
    data["previous_donations"],
    0
)

# Calculate 'donation_recency_score' (higher score for more recent donations)
data["donation_recency_score"] = (
    1 / (1 + data["days_since_last_donation"])
)

# Calculate 'event_engagement' score using max value for normalization
data["event_engagement"] = (
    data["events_attended"] /
    data["events_attended"].max()
)

# Calculate 'newsletter_engagement' score using max value for normalization
data["newsletter_engagement"] = (
    data["newsletter_clicks"] /
    data["newsletter_clicks"].max()
)

# Calculate 'volunteer_engagement' score using max value for normalization
data["volunteer_engagement"] = (
    data["volunteer_events"] /
    data["volunteer_events"].max()
)

# Calculate 'interaction_recency' score, normalizing to a 0-1 range and clipping
data["interaction_recency"] = (
    1 -
    data["days_since_last_interaction"] /
    data["days_since_last_interaction"].max()
).clip(0, 1)

# Calculate a composite 'engagement_score'
data["engagement_score"] = (
    0.30 * data["event_engagement"]
    + 0.30 * data["email_open_rate"]
    + 0.15 * data["newsletter_engagement"]
    + 0.15 * data["volunteer_engagement"]
    + 0.10 * data["interaction_recency"]
) * 100

print("✅ Feature engineering completed!")  # Confirm completion

print("\nNew dataset shape:")  # Display new dataset shape
print(data.shape)

display(data.head())  # Display the first few rows with new features

## CELL 5 — Define X and y

This cell separates the dataset into features (`X`) and the target variable (`y`). The `alumni_id` (identifier) and the target variable itself are removed from the features.

In [ ]:
# Separate features (X) from the target variable (y)
# Drop 'alumni_id' as it's an identifier and 'donated_next_12_months' as it's the target
X = data.drop(
    columns=[
        "alumni_id",
        "donated_next_12_months"
    ]
)

y = data["donated_next_12_months"]  # Assign the target variable

print("Features shape (X):", X.shape)  # Display the shape of the features DataFrame
print("Target shape (y):", y.shape)  # Display the shape of the target Series

## CELL 6 — Identify Numerical and Categorical Features

This cell automatically identifies and lists the numerical and categorical features in the dataset, which is crucial for applying appropriate preprocessing steps.

In [ ]:
# Identify numerical features (integers and floats)
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical features (object type)
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")  # Header for numerical features
print(numeric_features)  # Display the list of numerical features

print("\nCategorical Features:")  # Header for categorical features
print(categorical_features)  # Display the list of categorical features

## CELL 7 — Train/Test Split

This cell splits the dataset into training and testing sets. This is essential to evaluate the model's performance on unseen data and prevent overfitting.

In [ ]:
# Split the data into training and testing sets
# test_size=0.20 means 20% of data for testing, 80% for training
# random_state=42 for reproducibility
# stratify=y ensures that the proportion of target classes is the same in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])  # Display the number of training samples
print("Testing samples:", X_test.shape[0])  # Display the number of testing samples

## CELL 8 — Preprocessing

This cell defines a `ColumnTransformer` to apply different preprocessing steps to numerical and categorical features. Numerical features are scaled using `StandardScaler`, and categorical features are encoded using `OneHotEncoder`.

In [ ]:
# Create a ColumnTransformer for preprocessing numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",  # Name of the transformer for numerical features
            StandardScaler(),  # Apply StandardScaler to numerical features for normalization
            numeric_features  # List of numerical features
        ),
        (
            "cat",  # Name of the transformer for categorical features
            OneHotEncoder(
                handle_unknown="ignore"  # Apply OneHotEncoder to categorical features, ignoring unknown categories
            ),
            categorical_features  # List of categorical features
        )
    ]
)

print("✅ Preprocessing pipeline created successfully!")  # Confirm successful creation of the preprocessor

## CELL 9 — Logistic Regression

This cell constructs and trains a Logistic Regression model as part of a `Pipeline`. The pipeline first preprocesses the data using the defined `preprocessor` and then applies the Logistic Regression algorithm.

In [ ]:
# Create a pipeline for Logistic Regression: first preprocess, then apply the model
logistic_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the Logistic Regression model
        LogisticRegression(
            max_iter=1000,  # Set maximum iterations for convergence
            random_state=42 # For reproducibility
        )
    )
])

# Train the Logistic Regression model using the training data
logistic_model.fit(
    X_train,
    y_train
)

print("✅ Logistic Regression trained successfully!")  # Confirm successful training

## CELL 10 — Random Forest

This cell constructs and trains a Random Forest Classifier model as part of a `Pipeline`. Random Forest is an ensemble learning method that can handle complex relationships in the data.

In [ ]:
# Create a pipeline for Random Forest Classifier
rf_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the Random Forest model
        RandomForestClassifier(
            n_estimators=200,  # Number of trees in the forest
            random_state=42,  # Seed for reproducibility
            class_weight="balanced"  # Handle class imbalance by adjusting weights
        )
    )
])

# Train the Random Forest model using the training data
rf_model.fit(
    X_train,
    y_train
)

print("✅ Random Forest trained successfully!")  # Confirm successful training

## CELL 11 — XGBoost

This cell constructs and trains an XGBoost Classifier model within a `Pipeline`. XGBoost is a powerful gradient boosting framework known for its performance and speed.

In [ ]:
# Create a pipeline for XGBoost Classifier
xgb_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the XGBoost model
        XGBClassifier(
            n_estimators=200,  # Number of boosting rounds
            max_depth=5,  # Maximum tree depth
            learning_rate=0.05,  # Step size shrinkage to prevent overfitting
            random_state=42,  # Seed for reproducibility
            eval_metric="logloss",  # Evaluation metric for optimization
            use_label_encoder=False # Suppress warning for deprecated label encoder
        )
    )
])

# Train the XGBoost model using the training data
xgb_model.fit(
    X_train,
    y_train
)

print("✅ XGBoost trained successfully!")  # Confirm successful training

## CELL 12 — CatBoost

This cell constructs and trains a CatBoost Classifier model within a `Pipeline`. CatBoost is another powerful gradient boosting library that handles categorical features automatically and effectively.

In [ ]:
# Create a pipeline for CatBoost Classifier
cat_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the CatBoost model
        CatBoostClassifier(
            iterations=300,  # Number of boosting iterations (trees)
            depth=6,  # Depth of the trees
            learning_rate=0.05,  # Step size shrinkage
            verbose=False,  # Suppress training output
            random_seed=42  # Seed for reproducibility
        )
    )
])

# Train the CatBoost model using the training data
cat_model.fit(
    X_train,
    y_train
)

print("✅ CatBoost trained successfully!")  # Confirm successful training

## CELL 13 — Model Evaluation

This cell evaluates all trained models using various classification metrics to compare their performance on the test set. The results are compiled into a DataFrame for easy comparison.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluates a given machine learning model and returns various metrics."""

    # Make predictions on the test data
    y_pred = model.predict(X_test)

    # Get prediction probabilities for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate various evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Return metrics as a dictionary
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

results = []  # Initialize an empty list to store evaluation results

# Evaluate each model and store its results
results.append(evaluate_model(logistic_model, X_test, y_test, "Logistic Regression"))
results.append(evaluate_model(rf_model, X_test, y_test, "Random Forest"))
results.append(evaluate_model(xgb_model, X_test, y_test, "XGBoost"))
results.append(evaluate_model(cat_model, X_test, y_test, "CatBoost"))

# Convert the list of results dictionaries into a pandas DataFrame for easy comparison
comparison = pd.DataFrame(results)

# Display the comparison DataFrame, sorted by 'ROC-AUC' in descending order
display(
    comparison.sort_values(
        by="ROC-AUC",
        ascending=False
    )
)

print("\n--- Model Selection --- ")
print("For this prototype, Logistic Regression is chosen due to its competitive results and better interpretability, making it suitable for demonstrating the core functionality of a donor propensity model.")

## CELL 14 — Propensity Prediction Example

This cell demonstrates how to use the selected Logistic Regression model to predict donation propensity for new alumni. It calculates the probability of donation, converts it to a percentage, and categorizes it into 'High', 'Medium', or 'Low' likelihood.

In [ ]:
# Select the best performing model (Logistic Regression for this prototype)
selected_model = logistic_model

print("--- Alumni Donation Propensity Prediction Example ---")

# Predict the donation probabilities for the first 5 samples from the test set
# Using X_test.iloc[:5] to represent new, unseen alumni data
sample_alumni_data = X_test.iloc[:5]

sample_probabilities = (
    selected_model.predict_proba(
        sample_alumni_data  # Select the first 5 test samples for demonstration
    )[:, 1]  # Get probabilities for the positive class (donated)
)

print("\nSample donation propensities (likelihood of donating in the next 12 months):")

# Categorize and print each sample's propensity
for i, probability in enumerate(sample_probabilities):
    propensity_score = round(probability * 100, 2)  # Convert to percentage and round

    if 80 <= propensity_score <= 100:
        category = "High"
        interpretation = "Likely to Donate"
    elif 50 <= propensity_score < 80:
        category = "Medium"
        interpretation = "Moderate Donation Likelihood"
    else:
        category = "Low"
        interpretation = "Lower Donation Likelihood"

    print(f"Alumni {i+1}: {propensity_score}% (Category: {category} - {interpretation})")

print("\nImportant Note: Propensity is an estimated likelihood, not a guarantee. It helps prioritize outreach but does not guarantee a donation.")

## CELL 15 — Save the Complete Model Pipeline

This critical step saves the entire machine learning pipeline, including the preprocessing steps and the trained Logistic Regression model, to a `.pkl` file. This ensures that the model can be deployed and used in the future without retraining, and that preprocessing is consistent.

In [ ]:
model_path = "/content/alumni_donor_model_pipeline.pkl"  # Define the path where the model will be saved

# Save the complete pipeline (preprocessor + Logistic Regression model) to the specified path using joblib
# It's crucial to save the entire pipeline to ensure consistent preprocessing when making new predictions.
joblib.dump(
    logistic_model,  # The complete pipeline object
    model_path
)

print("✅ Model pipeline saved successfully!")  # Confirm successful saving
print("File saved at:", model_path)  # Print the path where the model was saved

## CELL 16 — Verify the PKL File

This cell verifies that the model pipeline file (`alumni_donor_model_pipeline.pkl`) was successfully created and exists in the `/content` directory of the Colab environment. It also displays the file size.

In [ ]:
import os  # Import the os module for interacting with the operating system

file_name = "/content/alumni_donor_model_pipeline.pkl" # Define the file path

# Check if the file exists
file_exists = os.path.exists(file_name)
print(f"File exists: {file_exists}")

# If the file exists, display its size
if file_exists:
    file_size_bytes = os.path.getsize(file_name)
    file_size_kb = round(file_size_bytes / 1024, 2)
    print(f"File size: {file_size_kb} KB")
else:
    print("File not found.")

## CELL 17 — Load the Saved PKL and Test It

This cell demonstrates loading the saved model pipeline and performing a test prediction to ensure that the saved file is valid and functional.

In [ ]:
# Load the saved model pipeline from the specified path using joblib
loaded_model = joblib.load(
    "/content/alumni_donor_model_pipeline.pkl"
)

print("✅ Model loaded successfully!")  # Confirm successful loading
print("Type of loaded model:", type(loaded_model)) # Print the type of the loaded object

# Use the loaded model to make a prediction on a sample from the test set
# This verifies that the loaded pipeline can preprocess data and make predictions.
sample_prediction = loaded_model.predict(X_test.iloc[[0]])

print(f"\nTest prediction for first sample (0=No Donation, 1=Donation): {sample_prediction[0]}")

# You can also get the probability
sample_proba = loaded_model.predict_proba(X_test.iloc[[0]])[:, 1]
print(f"Test probability for first sample: {round(sample_proba[0]*100, 2)}%")

## CELL 18 — Download the PKL File

This cell uses Google Colab's built-in functionality to download the trained model pipeline (`alumni_donor_model_pipeline.pkl`) to your local machine.

In [ ]:
from google.colab import files  # Import the files module for download functionality

# Initiate the download of the specified model file to your local machine
files.download(
    "/content/alumni_donor_model_pipeline.pkl"
)

print("✅ Model pipeline download initiated!")

In [ ]:
# Install necessary libraries for machine learning, including XGBoost and CatBoost
!pip install -q xgboost catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd  # Used for data manipulation and analysis
import numpy as np  # Used for numerical operations, especially with arrays

from sklearn.model_selection import train_test_split  # To split data into training and testing sets
from sklearn.compose import ColumnTransformer  # To apply different transformations to different columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # For scaling numerical features and encoding categorical features
from sklearn.pipeline import Pipeline  # To create a pipeline of transformations and a model

from sklearn.linear_model import LogisticRegression  # Logistic Regression model
from sklearn.ensemble import RandomForestClassifier  # Random Forest Classifier model

from xgboost import XGBClassifier  # XGBoost Classifier model
from catboost import CatBoostClassifier  # CatBoost Classifier model

from sklearn.metrics import (  # Various metrics for model evaluation
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import joblib  # Used for saving and loading Python objects, especially scikit-learn models

In [ ]:
from google.colab import files  # Import the files module for uploading files in Colab

uploaded = files.upload()  # Prompt user to upload the dataset file

KeyboardInterrupt: 

In [ ]:
uploaded = files.upload()

In [ ]:
from google.colab import files  # Import the files module for uploading files in Colab
import pandas as pd  # Import pandas for data manipulation

print("Please upload alumni_dataset.csv")  # Instruct the user to upload the specific file

uploaded = files.upload()  # Initiate the file upload dialog

# Get the name of the first uploaded file
file_name = next(iter(uploaded))

# Read the uploaded CSV file into a pandas DataFrame
data = pd.read_csv(file_name)

print("\nDataset loaded successfully!")  # Confirm successful loading
print("File name:", file_name)  # Display the name of the loaded file
print("Dataset shape:", data.shape)  # Display the dimensions of the DataFrame

display(data.head())  # Display the first 5 rows of the DataFrame to preview the data

In [ ]:
file_name = next(iter(uploaded))  # Get the name of the first uploaded file

data = pd.read_csv(file_name)  # Read the uploaded CSV file into a pandas DataFrame

print("Dataset loaded successfully!")  # Confirm successful loading
print("File name:", file_name)  # Display the name of the loaded file
print("Dataset shape:", data.shape)  # Display the dimensions of the DataFrame

display(data.head())  # Display the first 5 rows of the DataFrame to preview the data

Dataset loaded successfully!
File name: alumni_dataset.csv
Dataset shape: (5000, 16)


,alumni_id,graduation_year,age,events_attended,emails_received,emails_opened,newsletter_clicks,volunteer_events,previous_donations,total_donation_amount,donation_frequency,days_since_last_donation,career_level,industry,days_since_last_interaction,donated_next_12_months
0,10001,2018,40,4,40,7,6,3,4,1212,0,439,Executive,Finance,374,1
1,10002,2008,34,10,6,0,12,3,2,2768,3,211,Senior,Other,387,1
2,10003,1994,47,3,40,8,8,5,3,5776,2,399,Senior,Other,167,1
3,10004,2022,35,7,12,27,3,3,2,444,3,673,Mid,Education,479,1
4,10005,1987,52,3,49,1,14,3,2,9589,0,716,Entry,Healthcare,95,1


In [ ]:
print("Missing Values:")  # Header for missing values check
print(data.isnull().sum().sum())  # Calculate and print the total count of missing values in the DataFrame

print("\nDuplicate Rows:")  # Header for duplicate rows check
print(data.duplicated().sum())  # Calculate and print the total count of duplicate rows in the DataFrame

print("\nTarget Distribution:")  # Header for target variable distribution
print(data["donated_next_12_months"].value_counts())  # Show the distribution of values in the 'donated_next_12_months' column

Missing Values:
0

Duplicate Rows:
0

Target Distribution:
donated_next_12_months
1    4369
0     631
Name: count, dtype: int64


In [ ]:
# Calculate 'years_since_graduation' based on 'graduation_year'
data["years_since_graduation"] = (
    2026 - data["graduation_year"]
)

# Calculate 'email_open_rate', handling division by zero
data["email_open_rate"] = np.where(
    data["emails_received"] > 0,
    data["emails_opened"] / data["emails_received"],
    0
)

# Clip 'email_open_rate' to be between 0 and 1
data["email_open_rate"] = (
    data["email_open_rate"].clip(0, 1)
)

# Calculate 'average_donation_amount', handling division by zero
data["average_donation_amount"] = np.where(
    data["previous_donations"] > 0,
    data["total_donation_amount"] /
    data["previous_donations"],
    0
)

# Calculate 'donation_recency_score' (higher score for more recent donations)
data["donation_recency_score"] = (
    1 / (1 + data["days_since_last_donation"])
)

# Calculate 'event_engagement' score
data["event_engagement"] = (
    data["events_attended"] / 10
)

# Calculate 'newsletter_engagement' score
data["newsletter_engagement"] = (
    data["newsletter_clicks"] / 15
)

# Calculate 'volunteer_engagement' score
data["volunteer_engagement"] = (
    data["volunteer_events"] / 5
)

# Calculate 'interaction_recency' score, normalizing to a 0-1 range
data["interaction_recency"] = (
    1 -
    data["days_since_last_interaction"] / 500
)

# Clip 'interaction_recency' to be between 0 and 1
data["interaction_recency"] = (
    data["interaction_recency"].clip(0, 1)
)

# Calculate a composite 'engagement_score'
data["engagement_score"] = (
    0.30 * data["event_engagement"]
    + 0.30 * data["email_open_rate"]
    + 0.15 * data["newsletter_engagement"]
    + 0.15 * data["volunteer_engagement"]
    + 0.10 * data["interaction_recency"]
) * 100

print("Feature engineering completed!")  # Confirm completion

print("\nNew dataset shape:")  # Display new dataset shape
print(data.shape)

display(data.head())  # Display the first few rows with new features

Feature engineering completed!

New dataset shape:
(5000, 25)


,alumni_id,graduation_year,age,events_attended,emails_received,emails_opened,newsletter_clicks,volunteer_events,previous_donations,total_donation_amount,...,donated_next_12_months,years_since_graduation,email_open_rate,average_donation_amount,donation_recency_score,event_engagement,newsletter_engagement,volunteer_engagement,interaction_recency,engagement_score
0,10001,2018,40,4,40,7,6,3,4,1212,...,1,8,0.175000,303.000000,0.002273,0.4,0.400000,0.6,0.252,34.770000
1,10002,2008,34,10,6,0,12,3,2,2768,...,1,18,0.000000,1384.000000,0.004717,1.0,0.800000,0.6,0.226,53.260000
2,10003,1994,47,3,40,8,8,5,3,5776,...,1,32,0.200000,1925.333333,0.002500,0.3,0.533333,1.0,0.666,44.660000
3,10004,2022,35,7,12,27,3,3,2,444,...,1,4,1.000000,222.000000,0.001484,0.7,0.200000,0.6,0.042,63.420000
4,10005,1987,52,3,49,1,14,3,2,9589,...,1,39,0.020408,4794.500000,0.001395,0.3,0.933333,0.6,0.810,40.712245


In [ ]:
# Separate features (X) from the target variable (y)
# Drop 'alumni_id' as it's an identifier and 'donated_next_12_months' as it's the target
X = data.drop(
    columns=[
        "alumni_id",
        "donated_next_12_months"
    ]
)

y = data["donated_next_12_months"]  # Assign the target variable

print("Features shape:", X.shape)  # Display the shape of the features DataFrame
print("Target shape:", y.shape)  # Display the shape of the target Series

Features shape: (5000, 23)
Target shape: (5000,)


In [ ]:
# Identify numerical features (integers and floats)
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical features (object type)
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")  # Header for numerical features
print(numeric_features)  # Display the list of numerical features

print("\nCategorical Features:")  # Header for categorical features
print(categorical_features)  # Display the list of categorical features

Numerical Features:
['graduation_year', 'age', 'events_attended', 'emails_received', 'emails_opened', 'newsletter_clicks', 'volunteer_events', 'previous_donations', 'total_donation_amount', 'donation_frequency', 'days_since_last_donation', 'days_since_last_interaction', 'years_since_graduation', 'email_open_rate', 'average_donation_amount', 'donation_recency_score', 'event_engagement', 'newsletter_engagement', 'volunteer_engagement', 'interaction_recency', 'engagement_score']

Categorical Features:
['career_level', 'industry']


In [ ]:
# Split the data into training and testing sets
# test_size=0.20 means 20% of data for testing, 80% for training
# random_state=42 for reproducibility
# stratify=y ensures that the proportion of target classes is the same in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])  # Display the number of training samples
print("Testing samples:", X_test.shape[0])  # Display the number of testing samples

Training samples: 4000
Testing samples: 1000


In [ ]:
# Create a ColumnTransformer for preprocessing numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",  # Name of the transformer for numerical features
            StandardScaler(),  # Apply StandardScaler to numerical features for normalization
            numeric_features  # List of numerical features
        ),
        (
            "cat",  # Name of the transformer for categorical features
            OneHotEncoder(
                handle_unknown="ignore"  # Apply OneHotEncoder to categorical features, ignoring unknown categories
            ),
            categorical_features  # List of categorical features
        )
    ]
)

print("Preprocessing pipeline created successfully!")  # Confirm successful creation of the preprocessor

Preprocessing pipeline created successfully!


In [ ]:
# Create a pipeline for Logistic Regression: first preprocess, then apply the model
logistic_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the Logistic Regression model
        LogisticRegression(
            max_iter=1000  # Set maximum iterations for convergence
        )
    )
])

# Train the Logistic Regression model using the training data
logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully!")  # Confirm successful training

Logistic Regression trained successfully!


In [ ]:
# Create a pipeline for Random Forest Classifier
rf_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the Random Forest model
        RandomForestClassifier(
            n_estimators=200,  # Number of trees in the forest
            random_state=42,  # Seed for reproducibility
            class_weight="balanced"  # Handle class imbalance by adjusting weights
        )
    )
])

# Train the Random Forest model using the training data
rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully!")  # Confirm successful training

Random Forest trained successfully!


In [ ]:
# Create a pipeline for XGBoost Classifier
xgb_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the XGBoost model
        XGBClassifier(
            n_estimators=200,  # Number of boosting rounds
            max_depth=5,  # Maximum tree depth
            learning_rate=0.05,  # Step size shrinkage to prevent overfitting
            random_state=42,  # Seed for reproducibility
            eval_metric="logloss"  # Evaluation metric for optimization
        )
    )
])

# Train the XGBoost model using the training data
xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost trained successfully!")  # Confirm successful training

XGBoost trained successfully!


In [ ]:
# Create a pipeline for CatBoost Classifier
cat_model = Pipeline([
    (
        "preprocessor",  # Step 1: Apply the preprocessing transformations
        preprocessor
    ),
    (
        "model",  # Step 2: Apply the CatBoost model
        CatBoostClassifier(
            iterations=300,  # Number of boosting iterations (trees)
            depth=6,  # Depth of the trees
            learning_rate=0.05,  # Step size shrinkage
            verbose=False,  # Suppress training output
            random_seed=42  # Seed for reproducibility
        )
    )
])

# Train the CatBoost model using the training data
cat_model.fit(
    X_train,
    y_train
)

print("CatBoost trained successfully!")  # Confirm successful training

CatBoost trained successfully!


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluates a given machine learning model and prints various metrics."""

    # Make predictions on the test data
    y_pred = model.predict(X_test)

    # Get prediction probabilities for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate accuracy score
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    # Calculate precision score
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0  # Handle cases where no positive predictions are made
    )

    # Calculate recall score
    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0  # Handle cases where no actual positive instances exist
    )

    # Calculate F1 score
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0  # Handle cases with no positive predictions/actuals
    )

    # Calculate ROC-AUC score
    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    print("\n" + "=" * 60)  # Print a separator
    print(model_name)  # Print the model's name
    print("=" * 60)  # Print a separator

    # Print all calculated metrics, rounded to 4 decimal places
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))

    print("\nClassification Report:")  # Header for classification report
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0  # Generate a detailed classification report
        )
    )

    # Return metrics as a dictionary
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

In [ ]:
results = []  # Initialize an empty list to store evaluation results

# Evaluate Logistic Regression model and append results
results.append(
    evaluate_model(
        logistic_model,
        X_test,
        y_test,
        "Logistic Regression"
    )
)

# Evaluate Random Forest model and append results
results.append(
    evaluate_model(
        rf_model,
        X_test,
        y_test,
        "Random Forest"
    )
)

# Evaluate XGBoost model and append results
results.append(
    evaluate_model(
        xgb_model,
        X_test,
        y_test,
        "XGBoost"
    )
)

# Evaluate CatBoost model and append results
results.append(
    evaluate_model(
        cat_model,
        X_test,
        y_test,
        "CatBoost"
    )
)


Logistic Regression
Accuracy : 0.874
Precision: 0.874
Recall   : 1.0
F1 Score : 0.9328
ROC-AUC  : 0.6733

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       126
           1       0.87      1.00      0.93       874

    accuracy                           0.87      1000
   macro avg       0.44      0.50      0.47      1000
weighted avg       0.76      0.87      0.82      1000


Random Forest
Accuracy : 0.874
Precision: 0.8747
Recall   : 0.9989
F1 Score : 0.9327
ROC-AUC  : 0.6642

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.01      0.02       126
           1       0.87      1.00      0.93       874

    accuracy                           0.87      1000
   macro avg       0.69      0.50      0.47      1000
weighted avg       0.83      0.87      0.82      1000


XGBoost
Accuracy : 0.873
Precision: 0.8754
Recall   : 0.9966
F1 Score : 0.932
ROC-AUC  

In [ ]:
# Convert the list of results dictionaries into a pandas DataFrame for easy comparison
comparison = pd.DataFrame(results)

# Display the comparison DataFrame, sorted by 'ROC-AUC' in descending order
# This helps in quickly identifying the best performing model based on ROC-AUC
display(
    comparison.sort_values(
        by="ROC-AUC",
        ascending=False
    )
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.874,0.874000,1.000000,0.932764,0.673277
3,CatBoost,0.873,0.874624,0.997712,0.932122,0.671007
2,XGBoost,0.873,0.875377,0.996568,0.932049,0.664505
1,Random Forest,0.874,0.874749,0.998856,0.932692,0.664169


In [ ]:
model_path = "/content/alumni_donor_model_pipeline.pkl"  # Define the path where the model will be saved

# Save the best performing model (logistic_model in this case) to the specified path using joblib
joblib.dump(
    logistic_model,
    model_path
)

print("Model pipeline saved successfully!")  # Confirm successful saving
print("File:", model_path)  # Print the path where the model was saved

Model pipeline saved successfully!
File: /content/alumni_donor_model_pipeline.pkl


In [ ]:
# Load the saved model pipeline from the specified path using joblib
loaded_model = joblib.load(
    "/content/alumni_donor_model_pipeline.pkl"
)

print("Model loaded successfully!")  # Confirm successful loading

Model loaded successfully!


In [ ]:
# Predict the donation probabilities for the first 5 samples from the test set
sample_probabilities = (
    loaded_model.predict_proba(
        X_test.iloc[:5]  # Select the first 5 test samples
    )[:, 1]  # Get probabilities for the positive class (donated)
)

print("Sample donation probabilities:")  # Header for sample probabilities

# Print each sample's probability as a percentage
for probability in sample_probabilities:
    print(
        round(probability * 100, 2),  # Convert to percentage and round to 2 decimal places
        "%"
    )

Sample donation probabilities:
61.41 %
92.85 %
71.38 %
89.68 %
85.82 %


In [ ]:
import os  # Import the os module for interacting with the operating system

print(os.listdir("/content"))  # List all files and directories within the /content directory

['.config', 'sample_data']


In [ ]:
from google.colab import files  # Import the files module for uploading files in Colab

uploaded = files.upload()  # Prompt user to upload the dataset file

In [ ]:
print("logistic_model" in globals())  # Check if the 'logistic_model' variable exists in the global scope

False


In [ ]:
from google.colab import files

print("Please upload alumni_dataset.csv")

uploaded = files.upload()

Please upload alumni_dataset.csv


Saving alumni_dataset.csv to alumni_dataset (1).csv


In [ ]:
import pandas as pd  # Import pandas to use pd.read_csv

file_name = next(iter(uploaded))  # Get the name of the first uploaded file

data = pd.read_csv(file_name)  # Read the uploaded CSV file into a pandas DataFrame

print("Dataset loaded successfully!")  # Confirm successful loading
print("Shape:", data.shape)  # Display the dimensions of the DataFrame

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd  # Used for data manipulation and analysis
import numpy as np  # Used for numerical operations, especially with arrays
import joblib  # Used for saving and loading Python objects, particularly scikit-learn models

# Import necessary modules from scikit-learn for model building
from sklearn.model_selection import train_test_split  # To split data into training and testing sets
from sklearn.compose import ColumnTransformer  # To apply different transformers to different columns
from sklearn.pipeline import Pipeline  # To create a sequence of data transformations and a final estimator
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # For scaling numerical features and encoding categorical features
from sklearn.linear_model import LogisticRegression  # The machine learning model to be used

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
from google.colab import files  # Import the files module from google.colab for file upload functionality

print("Please upload alumni_dataset.csv")  # Prompt the user to upload the dataset file

uploaded = files.upload()  # Initiate the file upload widget and store the uploaded file(s) information

Please upload alumni_dataset.csv


In [ ]:
file_name = next(iter(uploaded))  # Get the name of the first uploaded file from the 'uploaded' dictionary

data = pd.read_csv(file_name)  # Read the CSV file into a pandas DataFrame

print("✅ Dataset loaded successfully!")  # Confirmation message
print("File name:", file_name)  # Display the name of the uploaded file
print("Dataset shape:", data.shape)  # Display the shape (rows, columns) of the loaded DataFrame

display(data.head())  # Display the first 5 rows of the DataFrame

✅ Dataset loaded successfully!
File name: alumni_dataset (2).csv
Dataset shape: (5000, 16)


,alumni_id,graduation_year,age,events_attended,emails_received,emails_opened,newsletter_clicks,volunteer_events,previous_donations,total_donation_amount,donation_frequency,days_since_last_donation,career_level,industry,days_since_last_interaction,donated_next_12_months
0,10001,2018,40,4,40,7,6,3,4,1212,0,439,Executive,Finance,374,1
1,10002,2008,34,10,6,0,12,3,2,2768,3,211,Senior,Other,387,1
2,10003,1994,47,3,40,8,8,5,3,5776,2,399,Senior,Other,167,1
3,10004,2022,35,7,12,27,3,3,2,444,3,673,Mid,Education,479,1
4,10005,1987,52,3,49,1,14,3,2,9589,0,716,Entry,Healthcare,95,1


In [ ]:
# Calculate 'years_since_graduation' as the difference between a future year (2026) and graduation year
data["years_since_graduation"] = (
    2026 - data["graduation_year"]
)

# Calculate 'email_open_rate', handling cases where 'emails_received' is zero to prevent division by zero errors
data["email_open_rate"] = np.where(
    data["emails_received"] > 0,
    data["emails_opened"] / data["emails_received"],
    0
)

# Clip 'email_open_rate' to ensure values are between 0 and 1 (inclusive)
data["email_open_rate"] = (
    data["email_open_rate"].clip(0, 1)
)

# Calculate 'average_donation_amount', handling cases where 'previous_donations' is zero
data["average_donation_amount"] = np.where(
    data["previous_donations"] > 0,
    data["total_donation_amount"] /
    data["previous_donations"],
    0
)

# Calculate 'donation_recency_score', giving higher scores to more recent donations (smaller 'days_since_last_donation')
data["donation_recency_score"] = (
    1 / (1 + data["days_since_last_donation"])
)

# Calculate 'event_engagement' score by normalizing 'events_attended'
data["event_engagement"] = (
    data["events_attended"] / 10
)

# Calculate 'newsletter_engagement' score by normalizing 'newsletter_clicks'
data["newsletter_engagement"] = (
    data["newsletter_clicks"] / 15
)

# Calculate 'volunteer_engagement' score by normalizing 'volunteer_events'
data["volunteer_engagement"] = (
    data["volunteer_events"] / 5
)

# Calculate 'interaction_recency' score, normalizing to a 0-1 range (1 for recent, 0 for very old interactions)
data["interaction_recency"] = (
    1 -
    data["days_since_last_interaction"] / 500
)

# Clip 'interaction_recency' to ensure values are between 0 and 1 (inclusive)
data["interaction_recency"] = (
    data["interaction_recency"].clip(0, 1)
)

# Calculate a composite 'engagement_score' based on a weighted sum of various engagement metrics
data["engagement_score"] = (
    0.30 * data["event_engagement"]
    + 0.30 * data["email_open_rate"]
    + 0.15 * data["newsletter_engagement"]
    + 0.15 * data["volunteer_engagement"]
    + 0.10 * data["interaction_recency"]
) * 100

print("✅ Feature engineering completed!")  # Confirmation message for feature engineering
print("New dataset shape:", data.shape)  # Display the updated shape of the DataFrame

✅ Feature engineering completed!
New dataset shape: (5000, 25)


In [ ]:
# Define feature matrix X by dropping 'alumni_id' (identifier) and 'donated_next_12_months' (target variable)
X = data.drop(
    columns=[
        "alumni_id",
        "donated_next_12_months"
    ]
)

# Define target vector y as the 'donated_next_12_months' column
y = data["donated_next_12_months"]

print("X shape:", X.shape)  # Display the shape of the feature matrix X
print("y shape:", y.shape)  # Display the shape of the target vector y

X shape: (5000, 23)
y shape: (5000,)


In [ ]:
# Automatically identify numerical features in the dataset
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Automatically identify categorical (object type) features in the dataset
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")  # Header for numerical features list
print(numeric_features)  # Display the list of identified numerical features

print("\nCategorical features:")  # Header for categorical features list
print(categorical_features)  # Display the list of identified categorical features

Numerical features:
['graduation_year', 'age', 'events_attended', 'emails_received', 'emails_opened', 'newsletter_clicks', 'volunteer_events', 'previous_donations', 'total_donation_amount', 'donation_frequency', 'days_since_last_donation', 'days_since_last_interaction', 'years_since_graduation', 'email_open_rate', 'average_donation_amount', 'donation_recency_score', 'event_engagement', 'newsletter_engagement', 'volunteer_engagement', 'interaction_recency', 'engagement_score']

Categorical features:
['career_level', 'industry']


In [ ]:
# Split the dataset into training and testing sets
# X_train, y_train will be used for model training
# X_test, y_test will be used for model evaluation
# test_size=0.20 means 20% of data is allocated for testing
# random_state=42 ensures reproducibility of the split
# stratify=y ensures that the proportion of target classes is maintained in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))  # Display the number of records in the training set
print("Testing records:", len(X_test))  # Display the number of records in the testing set

Training records: 4000
Testing records: 1000


In [ ]:
# Create a ColumnTransformer to apply different preprocessing steps to different types of features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",  # Name for the numerical features transformer
            StandardScaler(),  # Apply StandardScaler to numerical features for standardization
            numeric_features  # List of numerical feature names
        ),
        (
            "cat",  # Name for the categorical features transformer
            OneHotEncoder(
                handle_unknown="ignore"  # Apply OneHotEncoder to categorical features, ignoring unseen categories in test set
            ),
            categorical_features  # List of categorical feature names
        )
    ]
)

print("✅ Preprocessing pipeline created!")  # Confirmation message for pipeline creation

✅ Preprocessing pipeline created!


In [ ]:
# Create a machine learning pipeline for Logistic Regression
# Step 1: 'preprocessor' applies scaling and one-hot encoding
# Step 2: 'model' is the Logistic Regression classifier
logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000  # Set maximum iterations for the solver to converge
        )
    )
])

# Train the Logistic Regression model using the preprocessed training data
logistic_model.fit(
    X_train,
    y_train
)

print("✅ Logistic Regression trained successfully!")  # Confirmation message for successful training

✅ Logistic Regression trained successfully!


In [ ]:
print("logistic_model" in globals())  # Check if the 'logistic_model' variable is defined in the current global scope

True


In [ ]:
from sklearn.metrics import (  # Import various evaluation metrics from scikit-learn
    accuracy_score,  # For overall prediction accuracy
    precision_score,  # For the proportion of true positive predictions
    recall_score,  # For the proportion of actual positives correctly identified
    f1_score,  # The harmonic mean of precision and recall
    roc_auc_score,  # For Area Under the Receiver Operating Characteristic Curve
    classification_report  # To generate a text report showing main classification metrics
)

print("✅ Evaluation metrics imported successfully!")  # Confirmation message

✅ Evaluation metrics imported successfully!


In [ ]:
# Make predictions on the test set using the trained Logistic Regression model
y_pred = logistic_model.predict(X_test)

# Get the probability estimates for the positive class (class 1) from the test set
y_probability = (
    logistic_model.predict_proba(X_test)[:, 1]
)

# Calculate accuracy score
accuracy = accuracy_score(
    y_test,
    y_pred
)

# Calculate precision score, handling cases where no positive predictions are made
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

# Calculate recall score, handling cases where no actual positive instances exist
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

# Calculate F1 score, handling cases with no positive predictions or actuals
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

# Calculate ROC-AUC score based on predicted probabilities
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

# Print all calculated evaluation metrics, rounded to 4 decimal places
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")  # Header for the detailed classification report
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0  # Generate the classification report
    )
)

Accuracy : 0.874
Precision: 0.874
Recall   : 1.0
F1 Score : 0.9328
ROC-AUC  : 0.6733

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       126
           1       0.87      1.00      0.93       874

    accuracy                           0.87      1000
   macro avg       0.44      0.50      0.47      1000
weighted avg       0.76      0.87      0.82      1000



In [ ]:
model_path = "/content/alumni_donor_model_pipeline.pkl"  # Define the file path for saving the model

# Save the trained Logistic Regression model pipeline to the specified file path using joblib
joblib.dump(
    logistic_model,
    model_path
)

print("✅ Model pipeline saved successfully!")  # Confirmation message

✅ Model pipeline saved successfully!


In [ ]:
import os  # Import the os module to interact with the operating system, specifically for file operations

print(
    "Model exists:",  # Print a descriptive label
    os.path.exists(  # Check if a file or directory exists at the given path
        "/content/alumni_donor_model_pipeline.pkl"  # The path to the saved model file
    )
)

Model exists: True


In [ ]:
from google.colab import files  # Import the files module from google.colab for file download functionality

files.download(  # Initiate the download of the specified file to the local machine
    "/content/alumni_donor_model_pipeline.pkl"  # The path to the model file to be downloaded
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# End of Notebook Analysis
This marks the end of the machine learning model development and evaluation. The best model has been saved and is available for download.

SyntaxError: invalid syntax (3721558335.py, line 2)

In [ ]:
logistic_model.fit(X_train, y_train)